In [67]:
import pandas as pd
from sklearn.neighbors import NearestNeighbors
import matplotlib as plt
import numpy as np

In [68]:
df = pd.read_csv("../data/DBScan/dbscan_clustered_0.15.csv")
df.shape

(1229374, 442)

In [69]:
cluster_modes = df[df['cluster_labels'] != -1].groupby('cluster_labels').agg(pd.Series.mode) # gets the mode of every value, similar to a centriod of kmodes 
cluster_means = df[df['cluster_labels'] != -1].groupby('cluster_labels').mean() #gets the prevalence of a feature per cluster
cluster_amount = len(set(df[df['cluster_labels'] != -1]["cluster_labels"]))
print("shape: ", df[df["cluster_labels"] != -1].shape)
print("number of clusters: ", cluster_amount)

shape:  (1213083, 442)
number of clusters:  24


In [70]:
cluster_modes.iloc[:,:5]

,properties_images,properties_notes,properties_orientation_data,properties_name,geometry_type
cluster_labels,,,,,
0,0.0,0.0,0.0,1.0,1.0
1,0.0,0.0,0.0,1.0,0.0
2,0.0,0.0,0.0,1.0,1.0
3,0.0,1.0,1.0,1.0,1.0
4,0.0,0.0,0.0,1.0,1.0
5,0.0,0.0,1.0,1.0,1.0
6,0.0,0.0,0.0,1.0,1.0
7,0.0,1.0,0.0,1.0,1.0
8,0.0,0.0,1.0,1.0,1.0


In [71]:
for cluster in cluster_modes.index:
    print(f"Cluster {cluster}: {df[df["cluster_labels"] == cluster].shape}")


Cluster 0: (1125394, 442)
Cluster 1: (12464, 442)
Cluster 2: (1084, 442)
Cluster 3: (6612, 442)
Cluster 4: (2991, 442)
Cluster 5: (9630, 442)
Cluster 6: (1284, 442)
Cluster 7: (1101, 442)
Cluster 8: (6652, 442)
Cluster 9: (16679, 442)
Cluster 10: (1677, 442)
Cluster 11: (3869, 442)
Cluster 12: (994, 442)
Cluster 13: (2697, 442)
Cluster 14: (1552, 442)
Cluster 15: (2582, 442)
Cluster 16: (1620, 442)
Cluster 17: (2307, 442)
Cluster 18: (2445, 442)
Cluster 19: (949, 442)
Cluster 20: (2852, 442)
Cluster 21: (1456, 442)
Cluster 22: (2211, 442)
Cluster 23: (1981, 442)


In [72]:
discriminating_features = []

for col in cluster_modes.columns:
    temp = set()
    for cluster in cluster_modes[col]:
        temp.add(cluster)
        
    if len(temp) > 1:
        discriminating_features.append(col)
       
print("Discriminating Features: ") 
for feature in discriminating_features:
    print(f"\t{feature}")

Discriminating Features: 
	properties_notes
	properties_orientation_data
	geometry_type
	geometry_coordinates
	properties_trace_trace_feature
	properties_trace_trace_quality
	properties_trace_trace_type
	properties_trace_contact_type
	properties_trace_intrusive_contact_type
	properties_other_features
	properties_orientation_type
	properties_orientation_strike
	properties_orientation_dip_direction
	properties_orientation_dip
	properties_orientation_label
	properties_orientation_quality
	properties_orientation_feature_type
	properties_trace_other_contact_type
	properties_rock_unit_unit_label_abbreviation
	properties_rock_unit_rock_type
	properties_custom_fields_Station
	properties_custom_fields_STATNUM
	properties_custom_fields_VARIANT
	properties_custom_fields_GENERATION
	properties_custom_fields_UTMX
	properties_custom_fields_UTMY
	properties_custom_fields_OBJECTID
	properties_custom_fields_AREA
	properties_custom_fields_PERIMETER
	properties_custom_fields_GEO_
	properties_custom_field

In [73]:
print(cluster_means[discriminating_features])

                properties_notes  properties_orientation_data  geometry_type  \
cluster_labels                                                                 
0                       0.205616                     0.298410            1.0   
1                       0.039554                     0.411666            0.0   
2                       0.042435                     0.023985            1.0   
3                       1.000000                     1.000000            1.0   
4                       0.000000                     0.000000            1.0   
5                       0.007061                     1.000000            1.0   
6                       0.145639                     0.158879            1.0   
7                       0.998183                     0.000000            1.0   
8                       0.079675                     1.000000            1.0   
9                       0.175550                     1.000000            1.0   
10                      0.395349        

In [74]:
global_mean = df[df['cluster_labels'] != -1].drop(columns='cluster_labels').mean() # presence in the entire dataset
deviation = cluster_means[discriminating_features] / global_mean[discriminating_features]

deviation

,properties_notes,properties_orientation_data,geometry_type,geometry_coordinates,properties_trace_trace_feature,properties_trace_trace_quality,properties_trace_trace_type,properties_trace_contact_type,properties_trace_intrusive_contact_type,properties_other_features,...,properties_rock_unit_era,properties_rock_unit_period,properties_rock_unit_epoch,properties_rock_unit_group_unit_type,properties_custom_fields_rock_type,properties_custom_fields_rock_class,properties_custom_fields_TYPE,properties_custom_fields_gid,properties_custom_fields_state,properties_custom_fields_county
cluster_labels,,,,,,,,,,,,,,,,,,,,,
0,0.995551,0.942232,1.010381,1.010381,0.942362,0.970868,0.940880,0.677796,0.177581,0.940976,...,0.0000,0.0000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,0.191512,1.299837,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,3.787028,...,0.0000,0.0000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,0.205464,0.075734,1.010381,1.010381,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0000,0.0000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,4.841796,3.157508,1.010381,1.010381,0.000000,0.000000,0.000000,0.000000,0.000000,13.715308,...,0.0000,0.0000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4,0.000000,0.000000,1.010381,1.010381,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0000,0.0000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
5,0.034189,3.157508,1.010381,1.010381,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0000,0.0000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
6,0.705153,0.501660,1.010381,1.010381,0.000000,0.000000,0.000000,0.000000,0.000000,2.051197,...,0.0000,0.0000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
7,4.833001,0.000000,1.010381,1.010381,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0000,0.0000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
8,0.385771,3.157508,1.010381,1.010381,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0000,0.0000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


In [75]:
significant_deviations = {}
cluster_defining_features = {}

for cluster in deviation.index:
    features = deviation.loc[cluster]
    significant = features[features >= 2].sort_values(ascending=False)
    significant_deviations[cluster] = significant

for cluster, features in significant_deviations.items():
    print(f"\nCluster {cluster} : {df[df["cluster_labels"] == cluster].shape[0]} samples")
    if len(features) == 0:
        print("\tNo significant features")
        cluster_defining_features[cluster] = []
    else:
        temp = []
        for feature, ratio in features.items():
            # print(f"\t{feature}: {ratio:.2f}x more present in cluster")
            print(f"\t{feature}")
            temp.append(feature)
        cluster_defining_features[cluster] = temp


Cluster 0 : 1125394 samples
	No significant features

Cluster 1 : 12464 samples
	properties_other_features

Cluster 2 : 1084 samples
	properties_orientation_feature_type
	properties_orientation_type
	properties_orientation_strike
	properties_orientation_dip_direction
	properties_orientation_dip
	properties_orientation_label
	properties_orientation_quality

Cluster 3 : 6612 samples
	properties_custom_fields_Station
	properties_custom_fields_STATNUM
	properties_custom_fields_VARIANT
	properties_custom_fields_GENERATION
	properties_custom_fields_UTMX
	properties_custom_fields_UTMY
	properties_other_features
	properties_notes
	properties_orientation_data

Cluster 4 : 2991 samples
	properties_custom_fields_GEO_MISC
	properties_custom_fields_PlateID
	properties_custom_fields_PERIMETER
	properties_custom_fields_GEO_
	properties_custom_fields_GEO_ID
	properties_custom_fields_GEO_TYPE
	properties_custom_fields_GEO_SYM
	properties_custom_fields_GEO_LINK
	properties_custom_fields_GEO_PAT
	proper

In [76]:
deviation[deviation.index == 0]

,properties_notes,properties_orientation_data,geometry_type,geometry_coordinates,properties_trace_trace_feature,properties_trace_trace_quality,properties_trace_trace_type,properties_trace_contact_type,properties_trace_intrusive_contact_type,properties_other_features,...,properties_rock_unit_era,properties_rock_unit_period,properties_rock_unit_epoch,properties_rock_unit_group_unit_type,properties_custom_fields_rock_type,properties_custom_fields_rock_class,properties_custom_fields_TYPE,properties_custom_fields_gid,properties_custom_fields_state,properties_custom_fields_county
cluster_labels,,,,,,,,,,,,,,,,,,,,,
0,0.995551,0.942232,1.010381,1.010381,0.942362,0.970868,0.94088,0.677796,0.177581,0.940976,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
def val_count(data):
    counter = 0
    for i in data.columns:
        
        # if i in ["properties_viewed_timestamp" ,"properties_modified_timestamp","properties_symbology_circleColor","properties_id","properties_symbology_lineColor","properties_symbology_lineWidth","properties_symbology_lineDasharray","properties_sed_strat_section_strat_section_id","properties_strat_section_id","properties_symbology_fillColor","properties_orientation_id","properties_custom_fields_osm_id","properties_orientation_modified_timestamp","properties_custom_fields_id","properties_orientation_unix_timestamp","properties_gps_accuracy","properties_altitude","properties_notesTimestamp"]: 
        
        print(counter)
        print(data[i].value_counts())
        counter += 1
        print("=========================="*5)
    print(data.shape)
    
val_count(df = df[df["cluster_labels"] == 3])

0
properties_images
0.0    6612
Name: count, dtype: int64
1
properties_notes
1.0    6612
Name: count, dtype: int64
2
properties_orientation_data
1.0    6612
Name: count, dtype: int64
3
properties_name
1.0    6612
Name: count, dtype: int64
4
geometry_type
1.0    6612
Name: count, dtype: int64
5
geometry_coordinates
1.0    6612
Name: count, dtype: int64
6
properties_samples
0.0    6612
Name: count, dtype: int64
7
properties_altitude_accuracy
0.0    6612
Name: count, dtype: int64
8
properties_lng
0.0    6612
Name: count, dtype: int64
9
properties_image_basemap
0.0    6612
Name: count, dtype: int64
10
properties_lat
0.0    6612
Name: count, dtype: int64
11
properties__3d_structures
0.0    6612
Name: count, dtype: int64
12
properties_trace_trace_feature
0.0    6612
Name: count, dtype: int64
13
properties_trace_trace_quality
0.0    6612
Name: count, dtype: int64
14
properties_trace_trace_type
0.0    6612
Name: count, dtype: int64
15
properties_trace_contact_type
0.0    6612
Name: count, dtyp